### Tools 
Models can reauest to call tools that perform taskd such as fetching  data from databse seraching the web and runing code . Tools are pairings of:

1. A schema, including the name of the tool , a description , and /or argument definitions (often a JSON Schema)
2. A function or coroutine to execute.


In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")


In [2]:
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. I need to explain the reasons behind their ability to mimic human speech. Let me start by recalling what I know about parrots. They are part of the Psittaciformes order, right? Known for their colorful feathers and intelligence. Now, talking is a learned behavior, not something innate like in some other birds.\n\nFirst, I should mention their vocal anatomy. Parrots have a syrinx, which is the vocal organ in birds. But how is it different from other birds? Maybe the structure allows for more complex sounds. Also, their tongue is more flexible, which helps in forming different sounds. That\'s probably important for mimicking human speech.\n\nThen there\'s the social aspect. Parrots are social creatures. In the wild, they communicate with each other using calls. Mimicking sounds might help them integrate into a flock. When they\'re kept as pets, they might mimic their owners to bond or to be part of the social group

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str) ->str:
    """ Get the weather at a location"""
    return f"It's a sunny in {location}"


model_with_tool = model.bind_tools([get_weather])



In [5]:
response = model_with_tool.invoke("What's the weather like in Boston")
print(response)
for tool_call in response.tool_calls:
    ## View tool calls made by the model
    print(f"Tool : {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. Boston is the location here. So I\'ll call the function with "Boston" as the location. Make sure the JSON is correctly formatted with the name and arguments. No other parameters are needed. Just pass "Boston" in the location field.\n', 'tool_calls': [{'id': 'qz3hgnwq3', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 153, 'total_tokens': 266, 'completion_time': 0.189123551, 'completion_tokens_details': {'reasoning_tokens': 89}, 'prompt_time': 0.006836285, 'prompt_tokens_details': None, 'queue_time': 0.288689171, 'total_time': 0.195959836}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_t

In [6]:
## Step 1: Model generates tool calls 
messages = [{"role":"user" , "content":"What's te weather in Boston ?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

## Step 2 : Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    ## Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)


## Step 3 : pass reault back to model fpr final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)

It's sunny in Boston today. There are no clouds or rain, so it's a great day to be outdoors!
